# 知识图谱问答系统 - 使用示例

本notebook演示如何使用知识图谱问答系统处理论文并进行问答。

## 1. 环境设置

In [ ]:
import sys
sys.path.insert(0, '..')

from dotenv import load_dotenv
load_dotenv()

print("✓ 环境初始化完成")

## 2. 处理论文

从PDF文件中提取知识并构建图谱。

In [ ]:
from src.qa.chain import KnowledgeGraphPipeline

# 初始化流程
pipeline = KnowledgeGraphPipeline()

# 处理论文（请替换为实际PDF路径）
pdf_path = "../data/papers/sample_paper.pdf"
result = pipeline.process_paper(pdf_path, max_chunks=30)

print(f"处理结果:")
print(f"  论文ID: {result['paper_id']}")
print(f"  实体: {result['entities_stored']}")
print(f"  关系: {result['relations_stored']}")
print(f"  向量: {result['vectors_stored']}")

## 3. 查看知识图谱统计

In [ ]:
from src.storage.neo4j_client import Neo4jClient

# 连接Neo4j
neo4j = Neo4jClient()

# 获取统计信息
stats = neo4j.get_statistics()

print(f"知识图谱统计:")
print(f"  总实体数: {stats['total_entities']}")
print(f"  总关系数: {stats['total_relations']}")
print(f"\n实体类型分布:")
for et in stats['entity_types']:
    print(f"    {et['name']}: {et['count']}")

## 4. 搜索实体

In [ ]:
# 关键词搜索
keyword = "neural network"
entities = neo4j.search_entities(keyword, limit=10)

print(f"搜索 '{keyword}':")
for e in entities:
    print(f"  - [{e['type']}] {e['name']}")

## 5. 查看实体邻居

In [ ]:
# 选择一个实体查看其邻居
entity_name = entities[0]['name'] if entities else "BERT"

subgraph = neo4j.get_entity_neighbors(entity_name, depth=2)

print(f"实体 '{entity_name}' 的邻居（2跳）:")
print(f"\n节点 ({len(subgraph['nodes'])}):")
for node in subgraph['nodes'][:10]:
    print(f"  - [{node['type']}] {node['name']}")

print(f"\n关系 ({len(subgraph['relationships'])}):")
for rel in subgraph['relationships'][:10]:
    print(f"  - {rel['from']} --[{rel['type']}]--> {rel['to']}")

## 6. 问答测试

In [ ]:
from src.qa.chain import KnowledgeGraphQAChain

# 初始化问答链
chain = KnowledgeGraphQAChain()

# 测试问题
questions = [
    "这篇论文使用了什么数据集？",
    "主要方法是什么？",
    "实验结果如何？"
]

for question in questions:
    print(f"\nQ: {question}")
    result = chain.ask(question, include_context=False)
    print(f"A: {result['answer'][:200]}...")
    print(f"  [实体: {result['total_entities']}, 关系: {result['total_relations']}]")

## 7. 混合检索演示

In [ ]:
from src.retrieval.hybrid_searcher import HybridSearcher
from src.storage.chroma_client import ChromaClient

# 初始化检索器
chroma = ChromaClient()
searcher = HybridSearcher(chroma, neo4j)

# 混合检索
query = "deep learning optimization methods"
result = searcher.search(query, top_k_vectors=5, graph_depth=2)

print(f"查询: {query}\n")
print(f"向量召回: {len(result['vector_results'])} 个")
print(f"图谱实体: {result['total_entities']} 个")
print(f"图谱关系: {result['total_relations']} 个")

# 格式化上下文
context = searcher.format_context(result)
print(f"\n格式化上下文:\n{context[:500]}...")

## 8. 清理资源

In [ ]:
# 关闭所有连接
chain.close()
pipeline.close()
neo4j.close()

print("✓ 所有资源已清理")

## 使用提示

1. **处理前检查**: 确保PDF文件可读取，文字可提取
2. **调整参数**: 根据论文长度调整`max_chunks`参数
3. **评估质量**: 查看抽取的实体和关系数量，评估知识抽取质量
4. **优化检索**: 如果问答效果不佳，尝试调整`graph_depth`参数
5. **迭代改进**: 根据测试结果调整实体类型定义和提示模板